## 🎯 Learning Objectives
* Understand the critical need for observability in complex agentic AI systems.
* Implement structured logging to capture key agent actions, decisions, and state changes.
* Utilize distributed tracing to visualize the end-to-end execution flow of agent tasks, including LLM calls and tool invocations.
* Identify and collect essential metrics for monitoring agent performance, cost, and reliability.
* Apply debugging strategies to diagnose and resolve issues in non-deterministic agent behaviors.


## AG02-L08: Observability and Debugging Agent Behavior

In the realm of traditional software, debugging often involves setting breakpoints, inspecting variables, and following a predictable execution path. However, agentic AI systems introduce a new layer of complexity. Their non-deterministic nature, emergent behaviors, reliance on external tools, and interaction with large language models (LLMs) make them notoriously difficult to understand, troubleshoot, and optimize. This is where **observability** becomes paramount.

### What is Observability for AI Agents?

Observability is the ability to infer the internal states of a system by examining its external outputs. For AI agents, this means understanding *why* an agent made a particular decision, *how* it processed information, *what* tools it used, and *what* its internal state was at any given moment. It's not just about knowing *if* something went wrong, but *why* and *where*.

Imagine an autonomous financial trading agent that suddenly starts making unprofitable trades. Without observability, you'd only see the bad trades. With it, you could trace back its decision-making process: Was the market data it received flawed? Did its LLM misinterpret a prompt? Did a tool call fail silently? Was its internal risk assessment state corrupted?

### The Three Pillars of Observability (2026 Perspective)

By 2026, the industry has largely converged on three core pillars for robust observability, now specifically tailored for AI agents:

1.  **Structured Logging**: This goes beyond simple print statements. Structured logs capture key events (e.g., agent initialized, tool called, LLM prompt sent, state updated) with rich, machine-readable metadata (timestamps, agent ID, task ID, step name, input/output data, duration, cost). This allows for easy filtering, aggregation, and analysis.

2.  **Distributed Tracing**: Agentic workflows often involve multiple steps, LLM calls, tool invocations, and even interactions between different agents. Tracing provides an end-to-end view of a single request or task's journey through the system. Each operation becomes a 'span' in a 'trace', showing its duration, dependencies, and associated metadata. This is crucial for understanding latency bottlenecks and pinpointing failures across complex, asynchronous operations.

3.  **Metrics & Monitoring**: Aggregated data points that provide a high-level view of system health and performance. Examples include:
    *   **Agent-specific metrics**: Task success rate, average task duration, number of tool calls per task, LLM token usage per task, cost per task.
    *   **LLM-specific metrics**: Latency of LLM calls, token usage (input/output), error rates, hallucination rates (if detectable).
    *   **Tool-specific metrics**: Tool invocation frequency, success rate, average response time.
    *   **System-level metrics**: CPU/memory usage, network latency, API call rates.

### Debugging Challenges Unique to AI Agents

*   **Non-determinism**: LLMs can produce different outputs for the same prompt, making reproducibility difficult.
*   **Emergent Behavior**: Complex interactions between simple rules can lead to unexpected outcomes.
*   **Prompt Engineering Issues**: Subtle changes in prompts can drastically alter agent behavior.
*   **Tool Integration Failures**: External APIs can be unreliable or return unexpected data.
*   **State Management**: Incorrectly managed internal state can lead to agents 


In [ ]:
import os
import json
import time
import logging
import uuid
from datetime import datetime

# OpenTelemetry imports for tracing
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.semconv.resource import ResourceAttributes

# --- 1. Setup Structured Logging --- 
# Configure Python's standard logging to output JSON for structured logs
class JsonFormatter(logging.Formatter):
    def format(self, record):
        log_record = {
            "timestamp": datetime.fromtimestamp(record.created).isoformat(),
            "level": record.levelname,
            "name": record.name,
            "message": record.getMessage(),
            "agent_id": getattr(record, 'agent_id', 'N/A'),
            "task_id": getattr(record, 'task_id', 'N/A'),
            "step": getattr(record, 'step', 'N/A'),
            "data": getattr(record, 'data', {})
        }
        return json.dumps(log_record)

# Get a logger instance
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Ensure only one handler is added to avoid duplicate logs
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(JsonFormatter())
    logger.addHandler(handler)

# --- 2. Setup OpenTelemetry Tracing --- 
# Resource for our service
resource = Resource.create({
    ResourceAttributes.SERVICE_NAME: "research-agent-service",
    ResourceAttributes.SERVICE_VERSION: "1.0.0",
})

# Configure TracerProvider
provider = TracerProvider(resource=resource)
# For demonstration, we'll export spans to the console
span_processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(span_processor)

# Set the global tracer provider
trace.set_tracer_provider(provider)

# Get a tracer instance
tracer = trace.get_tracer(__name__)

# --- Mock Components for our Agent --- 

class MockLLM:
    """Simulates an LLM call with logging and tracing."""
    def __init__(self, name="MockLLM"):
        self.name = name

    def generate(self, prompt: str, task_id: str, agent_id: str, step: str):
        with tracer.start_as_current_span(f"LLM_Call:{self.name}") as span:
            span.set_attribute("llm.model_name", self.name)
            span.set_attribute("llm.prompt_length", len(prompt))
            span.set_attribute("llm.task_id", task_id)
            span.set_attribute("llm.agent_id", agent_id)
            
            logger.info(
                "LLM Request", 
                extra={'agent_id': agent_id, 'task_id': task_id, 'step': step, 'data': {'prompt_snippet': prompt[:50] + '...'}}
            )
            
            # Simulate LLM processing time and token usage
            time.sleep(0.5)
            response = f"LLM response to: '{prompt[:30]}...'"
            input_tokens = len(prompt) // 4 # rough estimate
            output_tokens = len(response) // 4
            cost = (input_tokens * 0.00001) + (output_tokens * 0.00003) # mock cost

            span.set_attribute("llm.response_length", len(response))
            span.set_attribute("llm.input_tokens", input_tokens)
            span.set_attribute("llm.output_tokens", output_tokens)
            span.set_attribute("llm.cost_usd", cost)
            span.set_attribute("llm.success", True)

            logger.info(
                "LLM Response", 
                extra={'agent_id': agent_id, 'task_id': task_id, 'step': step, 'data': {'response_snippet': response[:50] + '...', 'input_tokens': input_tokens, 'output_tokens': output_tokens, 'cost_usd': cost}}
            )
            return response

class MockSearchTool:
    """Simulates a search tool call with logging and tracing."""
    def __init__(self, name="MockSearchTool"):
        self.name = name

    def search(self, query: str, task_id: str, agent_id: str, step: str):
        with tracer.start_as_current_span(f"Tool_Call:{self.name}") as span:
            span.set_attribute("tool.name", self.name)
            span.set_attribute("tool.query", query)
            span.set_attribute("tool.task_id", task_id)
            span.set_attribute("tool.agent_id", agent_id)

            logger.info(
                "Tool Request", 
                extra={'agent_id': agent_id, 'task_id': task_id, 'step': step, 'data': {'tool_name': self.name, 'query': query}}
            )
            
            # Simulate search time and result
            time.sleep(0.3)
            results = f"Search results for '{query}': Found 3 relevant documents. Summary: ..."

            span.set_attribute("tool.result_length", len(results))
            span.set_attribute("tool.success", True)

            logger.info(
                "Tool Response", 
                extra={'agent_id': agent_id, 'task_id': task_id, 'step': step, 'data': {'tool_name': self.name, 'results_snippet': results[:50] + '...'}}
            )
            return results

# --- The Agent Itself --- 

class ResearchAgent:
    """A simple agent that uses an LLM and a search tool to answer queries."""
    def __init__(self, agent_id: str):
        self.agent_id = agent_id
        self.llm = MockLLM()
        self.search_tool = MockSearchTool()
        logger.info(
            "Agent Initialized", 
            extra={'agent_id': self.agent_id, 'step': 'init', 'data': {'llm_model': self.llm.name, 'tool_name': self.search_tool.name}}
        )

    def run(self, query: str):
        task_id = str(uuid.uuid4())
        
        # Start a trace for the entire agent run
        with tracer.start_as_current_span("Agent_Run") as span:
            span.set_attribute("agent.id", self.agent_id)
            span.set_attribute("agent.task_id", task_id)
            span.set_attribute("agent.query", query)
            
            logger.info(
                "Agent Task Started", 
                extra={'agent_id': self.agent_id, 'task_id': task_id, 'step': 'start', 'data': {'query': query}}
            )

            # Step 1: LLM decides if a tool is needed
            decision_prompt = f"Given the query '{query}', do I need to use a search tool? Respond with 'YES' or 'NO' followed by reasoning."
            llm_decision = self.llm.generate(decision_prompt, task_id, self.agent_id, "llm_decision")
            
            use_tool = "YES" in llm_decision.upper()
            logger.info(
                "Agent Decision", 
                extra={'agent_id': self.agent_id, 'task_id': task_id, 'step': 'decision', 'data': {'llm_decision': llm_decision, 'use_tool': use_tool}}
            )

            context = ""
            if use_tool:
                # Step 2: Use the search tool
                search_query = f"Find information about {query}"
                search_results = self.search_tool.search(search_query, task_id, self.agent_id, "tool_search")
                context = f"Based on search results: {search_results}"
                logger.info(
                    "Tool Used", 
                    extra={'agent_id': self.agent_id, 'task_id': task_id, 'step': 'tool_used', 'data': {'search_query': search_query, 'context_snippet': context[:50] + '...'}}
                )
            else:
                logger.info(
                    "Tool Skipped", 
                    extra={'agent_id': self.agent_id, 'task_id': task_id, 'step': 'tool_skipped', 'data': {'reason': 'LLM decided not to use tool'}}
                )

            # Step 3: LLM synthesizes the final answer
            synthesis_prompt = f"Answer the query '{query}' using the following context: {context}. If no context, answer based on general knowledge."
            final_answer = self.llm.generate(synthesis_prompt, task_id, self.agent_id, "llm_synthesis")

            span.set_attribute("agent.final_answer_length", len(final_answer))
            span.set_attribute("agent.status", "completed")
            
            logger.info(
                "Agent Task Completed", 
                extra={'agent_id': self.agent_id, 'task_id': task_id, 'step': 'complete', 'data': {'final_answer_snippet': final_answer[:50] + '...'}}
            )
            return final_answer

# --- Run the Agent and Observe --- 

if __name__ == "__main__":
    print("\n--- Running Agent with Observability ---\n")
    agent = ResearchAgent(agent_id="researcher-001")
    
    query1 = "What are the key features of quantum computing in 2026?"
    print(f"\nQuery 1: {query1}\n")
    answer1 = agent.run(query1)
    print(f"\nFinal Answer 1: {answer1}\n")

    query2 = "Tell me a joke."
    print(f"\nQuery 2: {query2}\n")
    answer2 = agent.run(query2)
    print(f"\nFinal Answer 2: {answer2}\n")

    print("\n--- Observability demonstration complete ---\n")


### Interpreting the Output and Practical Implications

When you run the code above, you'll observe two distinct types of output:

1.  **Structured Logs (JSON lines)**: Each line represents a specific event within the agent's execution. Notice how each log entry is a JSON object containing `timestamp`, `level`, `message`, `agent_id`, `task_id`, `step`, and a `data` dictionary. This structured format is incredibly powerful:
    *   **Filtering**: You can easily filter logs by `agent_id`, `task_id`, `step` (e.g., all 'LLM Request' logs for a specific task). 
    *   **Analysis**: You can aggregate data points like `input_tokens`, `output_tokens`, `cost_usd` to understand resource consumption over time. 
    *   **Context**: The `data` field provides rich context specific to the event, helping you understand *what* happened (e.g., the prompt snippet sent to the LLM, the query sent to the tool).
    *   **Debugging**: If an agent fails, you can trace back through the logs for that `task_id` to see the exact sequence of events, inputs, and outputs leading up to the failure.

2.  **OpenTelemetry Traces (Console Output)**: After the JSON logs, you'll see output from the `ConsoleSpanExporter`. This represents the distributed trace. Each block starting with `Span(name='...')` is a 'span'.
    *   **Hierarchy**: Notice the indentation. The `Agent_Run` span is the parent, and `LLM_Call` and `Tool_Call` spans are its children. This shows the causal relationship and execution flow.
    *   **Duration**: Each span shows its start and end time, allowing you to identify bottlenecks (e.g., a slow LLM call or tool invocation).
    *   **Attributes**: The `attributes` field within each span provides key-value pairs of metadata. For example, `llm.model_name`, `llm.input_tokens`, `tool.query`, `agent.task_id`. These attributes are crucial for filtering traces and understanding the context of each operation.
    *   **Correlation**: The `trace_id` and `span_id` link related operations together. In a real system, these would be propagated across microservices, allowing you to trace a single request through an entire distributed architecture.

### Performance Trade-offs

Implementing comprehensive observability does come with overhead:

*   **Latency**: Emitting logs and spans takes CPU cycles and network bandwidth. For high-throughput systems, this can introduce measurable latency.
*   **Storage Costs**: Storing vast amounts of structured logs and trace data can be expensive, especially in cloud environments. 
*   **Complexity**: Integrating observability tools and maintaining dashboards adds operational overhead.

**Mitigation**: 
*   **Sampling**: For high-volume traces, only a percentage of traces might be sent to the backend. 
*   **Asynchronous Processing**: Logs and traces are often sent asynchronously to minimize impact on the main application thread. 
*   **Intelligent Logging**: Log only what's necessary at each level (e.g., `DEBUG` for development, `INFO` for production, `ERROR` for critical failures).

### Typical Use Cases and Beyond the Demo

This simple demonstration highlights the core concepts. In a production environment, these logs and traces would be sent to specialized observability platforms:

*   **AI-Specific Observability Platforms (e.g., Langfuse, Arize, Helicone)**: These platforms are purpose-built for LLM and agent workflows, offering features like prompt versioning, cost tracking, hallucination detection, and human-in-the-loop feedback integration.
*   **General Observability Platforms (e.g., Datadog, Honeycomb, Grafana, OpenSearch, Google Cloud Operations Suite)**: These provide robust solutions for ingesting, storing, visualizing, and alerting on logs, metrics, and traces from any application, including agentic systems.

**Key Use Cases:**

*   **Root Cause Analysis**: Quickly identify why an agent failed, hallucinated, or produced an incorrect output by examining the exact sequence of LLM calls, tool interactions, and internal states.
*   **Performance Optimization**: Pinpoint bottlenecks (e.g., slow LLM providers, inefficient tool calls) and optimize agent workflows for speed and cost.
*   **Cost Management**: Monitor token usage and API costs in real-time to manage budgets effectively.
*   **Behavioral Analysis**: Understand how agents make decisions, identify common patterns, and detect unexpected behaviors.
*   **Compliance & Auditing**: Maintain a verifiable record of agent actions for regulatory compliance or internal audits.
*   **A/B Testing & Experimentation**: Compare the performance and behavior of different agent versions or prompt strategies.

By embracing a robust observability strategy, architects and designers can build more reliable, performant, and understandable agentic AI systems, moving them from experimental prototypes to production-grade solutions.


### Resources

*   **OpenTelemetry Documentation**: The vendor-neutral standard for instrumenting applications for observability.
    *   [https://opentelemetry.io/docs/](https://opentelemetry.io/docs/)
    *   [OpenTelemetry Python SDK](https://opentelemetry.io/docs/languages/python/)

*   **LangChain Observability (LangSmith)**: A platform specifically designed for debugging, testing, evaluating, and monitoring LLM applications and agents.
    *   [https://python.langchain.com/docs/langsmith/](https://python.langchain.com/docs/langsmith/)

*   **Arize AI**: An ML observability platform that helps monitor, troubleshoot, and improve models in production.
    *   [https://www.arize.com/](https://www.arize.com/)

*   **Helicone**: An open-source platform for LLM observability and analytics.
    *   [https://www.helicone.ai/](https://www.helicone.ai/)

*   **Google Cloud Operations Suite (Logging, Monitoring, Tracing)**: Comprehensive cloud-native observability tools.
    *   [Google Cloud Logging](https://cloud.google.com/logging)
    *   [Google Cloud Monitoring](https://cloud.google.com/monitoring)
    *   [Google Cloud Trace](https://cloud.google.com/trace)

*   **Structured Logging Best Practices**: General guidance on effective logging.
    *   [12 Factor App - Logs](https://12factor.net/logs) (Principle X: Treat logs as event streams)
